In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Modul 11: Transformer

Cerita lengkapnya ada di `README.md`. Mode cepat menghitung attention pada matriks mini, lalu melatih model kecil pada kalimat sintetis. Tidak ada unduhan. Bukan data asli perusahaan.

Sebelum kode: jika satu baris bobot perhatian adalah 0.2, 0.3, dan 0.5, jumlahnya berapa?


In [2]:
# Satu ide: panggil numpy untuk matriks mini.
import numpy as np


In [3]:
# Satu ide: attention membuat bobot positif yang jumlah satu barisnya dekat ke 1.
def softmax(baris):
    geser = baris - baris.max(axis=-1, keepdims=True)
    e = np.exp(geser)
    return e / e.sum(axis=-1, keepdims=True)

q = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
k = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
skor = q @ k.T / np.sqrt(2)
bobot = softmax(skor)
selisih_jumlah = float(np.max(np.abs(bobot.sum(axis=1) - 1)))
print("selisih_jumlah", round(selisih_jumlah, 6))
assert selisih_jumlah <= 1e-6


selisih_jumlah 0.0


In [4]:
# Satu ide: nilai dibawa sesuai bobot perhatian.
v = np.array([[2.0, 0.0], [0.0, 3.0], [1.0, 1.0]])
campuran = bobot @ v
print("bentuk_campuran", tuple(campuran.shape))


bentuk_campuran (3, 2)


## Kalimat sintetis

Delapan kalimat ditulis di sini. Label 1 berarti ada kucing. Label 0 berarti cuaca. Bukan data asli perusahaan.


In [5]:
# Satu ide: siapkan kalimat sintetis berlabel.
kalimat = [
    "kucing lucu sekali",
    "kucing kecil tidur",
    "anak kucing lucu",
    "hujan deras turun",
    "angin kencang malam",
    "awan gelap hujan",
    "kucing duduk diam",
    "hujan deras angin",
]
jawaban = np.array([1, 1, 1, 0, 0, 0, 1, 0], dtype=float)
print("banyak", len(kalimat))


banyak 8


In [6]:
# Satu ide: ubah tiap kalimat menjadi kantung token.
kamus = {}
for teks in kalimat:
    for tok in teks.split():
        if tok not in kamus:
            kamus[tok] = len(kamus)

def kantung(teks):
    vektor = np.zeros(len(kamus))
    for tok in teks.split():
        vektor[kamus[tok]] += 1
    return vektor

matriks = np.stack([kantung(teks) for teks in kalimat])
print("kolom", int(matriks.shape[1]))


kolom 16


In [7]:
# Satu ide: ukur fungsi kerugian sebelum ada langkah belajar.
indeks_latih = [0, 1, 3, 4, 6]
pred0 = np.full(len(indeks_latih), 0.5)
kerugian_awal = float(np.mean((pred0 - jawaban[indeks_latih]) ** 2))
print("kerugian_awal", round(kerugian_awal, 6))


kerugian_awal 0.25


In [8]:
# Satu ide: tiga puluh langkah pada data latih sintetis.
bobot = np.zeros(matriks.shape[1])
geser = 0.0
laju = 0.5
for _ in range(30):
    pred = 1 / (1 + np.exp(-(matriks[indeks_latih] @ bobot + geser)))
    sisa = (pred - jawaban[indeks_latih]) / len(indeks_latih)
    bobot = bobot - laju * (matriks[indeks_latih].T @ sisa)
    geser = geser - laju * float(sisa.sum())
kerugian_akhir = float(np.mean((pred - jawaban[indeks_latih]) ** 2))
print("kerugian_akhir", round(kerugian_akhir, 6))


kerugian_akhir 0.008027


In [9]:
# Satu ide: salah pada data uji harus lebih kecil daripada baseline.
indeks_uji = [2, 5, 7]
pred_uji = 1 / (1 + np.exp(-(matriks[indeks_uji] @ bobot + geser)))
salah_uji_model = int(((pred_uji >= 0.5) != (jawaban[indeks_uji] >= 0.5)).sum())
salah_uji_baseline = int((np.ones(len(indeks_uji)) != jawaban[indeks_uji]).sum())
print("salah_uji_model", salah_uji_model)
print("salah_uji_baseline", salah_uji_baseline)
assert salah_uji_baseline - salah_uji_model >= 1


salah_uji_model 0
salah_uji_baseline 2


In [10]:
# Satu ide: mode penuh memakai revisi yang dipin, dan mode cepat tidak mengunduh.
import importlib.util
import sys
from pathlib import Path
akar = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "phase2-ai" / "muat_hf.py").is_file())
sys.path.insert(0, str(akar / "phase2-ai"))
from muat_hf import REVISI
spec = importlib.util.spec_from_file_location("muat11", akar / "phase2-ai" / "modul-11-attention-transformer" / "muat.py")
muat = importlib.util.module_from_spec(spec)
spec.loader.exec_module(muat)
print("model_penuh", muat.ID_PENUH)
print("revisi_penuh", REVISI[muat.ID_PENUH])
print("model_opsional", muat.ID_OPSIONAL)
print("revisi_opsional", REVISI[muat.ID_OPSIONAL])
print("unduh", False)


model_penuh google/bert_uncased_L-2_H-128_A-2
revisi_penuh 30b0a37ccaaa32f332884b96992754e246e48c5f
model_opsional distilbert/distilbert-base-uncased
revisi_opsional 12040accade4e8a0f71eabdb258fecc2e7e948be
unduh False
